Testing training/inference with classifier-free guidance

Constraints:

- vel should be <= 0.2
- acc should be <= 0.4

Since it's normalized to canvas size [0,1], we're going scale it up by 1.5x, so we need to scale down the vel/acc by 1.5x.  We originally had 0.3 and 0.6, which we scaled down to 0.2, 0.4

We'll use loss terms:
- relu[vel - vel_max]
- relu[acc - acc_max]

Assume trajectory is executed at 50Hz, so 0.02s per point.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# diffusion policy import
from typing import Tuple, Sequence, Dict, Union, Optional
import numpy as np
import torch
import torch.nn as nn
from diffusers.schedulers.scheduling_ddpm import DDPMScheduler
from diffusers.training_utils import EMAModel
from diffusers.optimization import get_scheduler

# Painting imports
import cv2
from style.diffusion_policy_gml.dataset import PushTStateDataset, GmlDataset, normalize_data
from style.diffusion_policy_gml.network import MemorizationModel, ConditionalUnet1D, compute_noise, compute_orig
from style.diffusion_policy_gml.env import PaintingEnv
import style.diffusion_policy_gml.network as network
from style.diffusion_policy_gml.utils import plot_traj
import toppra as ta
import toppra.constraint as constraint
import toppra.algorithm as algo

# General
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from pathlib import Path
from torch.utils.tensorboard import SummaryWriter
import gerry

In [ ]:
pred_horizon = 512
obs_horizon = 1  # pad end but don't pad start, to discourage standing still at start
action_horizon = 1
obs_dim = 0  # x/y
action_dim = 5  # dx/dy/penup

num_diffusion_iters = 100

## Load Dataset

In [ ]:
# dataset_path = "data/gml_000000.zarr"
# dataset_path = "data/gml_003000.zarr"
dataset_path1 = "data/gml_by_drawing_PRESERVE_ASPECT_CENTERED_003000.zarr"
dataset_path2 = "data/toppra_by_drawing_PRESERVE_ASPECT_CENTERED_003000.zarr"

class MergedDataset(torch.utils.data.Dataset):
    def __init__(self, datasets: Sequence[torch.utils.data.Dataset]):
        self.datasets = datasets
        self.lengths = [len(d) for d in datasets]
        self.cumsum = np.cumsum([0] + self.lengths)
    def __len__(self):
        return sum(self.lengths)
    def __getitem__(self, idx):
        for i, d in enumerate(self.datasets):
            if idx < self.cumsum[i+1]:
                return i, d[idx - self.cumsum[i]]
        raise IndexError

with gerry.Stopwatch("Loading dataset"):
    dataset1 = GmlDataset(
        dataset_path=dataset_path1,
        sequence_length=pred_horizon,
        pad_before=0,
        pad_after=0,
        # stride=10,
        action_delta=True,
        action_penlift=True,
        normalize=dict(obs=False, action=True),
        # max_drawings=100
    )
    dataset2 = GmlDataset(
        dataset_path=dataset_path2,
        sequence_length=pred_horizon,
        pad_before=0,
        pad_after=0,
        # stride=10,
        action_delta=True,
        action_penlift=True,
        normalize=dict(obs=False, action=True),
        # max_drawings=100
    )
    dataset_ = MergedDataset([dataset1, dataset2])


# create dataloader
with gerry.Stopwatch("Creating dataloader"):
    dataloader = torch.utils.data.DataLoader(
        dataset_,
        batch_size=128,
        num_workers=1,
        shuffle=True,
        pin_memory=True,
        persistent_workers=True
    )

# visualize data in batch
print("Num batches:          ", len(dataloader))
idx, batch = next(iter(dataloader))
print("idx:                  ", idx.shape)
print("batch['obs'].shape:   ", batch['obs'].shape)
print("batch['action'].shape:", batch['action'].shape)

In [ ]:
acts = dataset2.normalized_train_data['action'][:dataset2.episode_ends[0]]
obss = dataset2.normalized_train_data['obs'][:dataset2.episode_ends[0]]
print(f'{acts.shape=}   {obss.shape=}')

fig, axes = plt.subplots(2, 1, figsize=(6, 5))
axes[0].plot(obss, '.-', linewidth=0.5)
axes[1].plot(acts, '.-', linewidth=0.5)
axes[1].set_ylim(-2, 2);

In [ ]:
plot_traj(plt.gca(), acts, x0=obss[0])
plt.axis('equal');

## Diffusion Setup

In [ ]:
# Noise scheduler
noise_scheduler = DDPMScheduler(
    num_train_timesteps=num_diffusion_iters,
    # the choise of beta schedule has big impact on performance
    # we found squared cosine works the best
    beta_schedule='squaredcos_cap_v2',
    # clip output to [-1,1] to improve stability
    clip_sample=True,
    clip_sample_range=5,
    # our network predicts noise (instead of denoised action)
    prediction_type='epsilon'
)

In [ ]:
# Network!
noise_pred_net = ConditionalUnet1D(
    input_dim=action_dim,
    global_cond_dim=1,
)

def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_normal_(m.weight)
        nn.init.zeros_(m.bias)
noise_pred_net.apply(init_weights);

In [ ]:
# Test with example inputs
noised_action = torch.randn((1, pred_horizon, action_dim))
obs = torch.zeros((1, 1))
diffusion_iter = torch.zeros((1,), dtype=torch.long)

# compute
noise = noise_pred_net(
    sample=noised_action,
    timestep=diffusion_iter,
    global_cond=obs)

# check denoising
denoised_action = noised_action - noise

# device transfer
device = torch.device('cuda')
_ = noise_pred_net.to(device)

## Training

In [ ]:
num_epochs = 500 // len(dataloader) + 1
num_epochs = 1500 // len(dataloader) + 1

ema = EMAModel(
    parameters=noise_pred_net.parameters(),
    model=noise_pred_net,
    power=0.75)

optimizer = torch.optim.AdamW(
    params=noise_pred_net.parameters(),
    lr=1e-4, weight_decay=1e-6)

lr_scheduler = get_scheduler(
    name='cosine',
    optimizer=optimizer,
    num_warmup_steps=500,
    num_training_steps=len(dataloader) * num_epochs
)

if True:
    noise_pred_net.apply(init_weights)
all_losses = list()

writer = SummaryWriter()  # Tensorboard

In [ ]:
try:
    starting_step = len(all_losses)
    with tqdm(range(num_epochs), desc='Epoch') as tglobal:
        # epoch loop
        for epoch_idx in tglobal:
            epoch_loss = list()
            with tqdm(dataloader, desc='Batch') as tdataloader:
                # batch loop
                for idx, batch_n in tdataloader:
                    # Extract data
                    obs_n = batch_n['obs'].to(device)
                    action_n = batch_n['action'].to(device)
                    action_n = torch.cat([obs_n, action_n], dim=-1)
                    B = obs_n.shape[0]
                    # assert obs_n.shape[1] == obs_horizon
                    # global_cond = obs_n.flatten(start_dim=1)
                    global_cond = idx.to(device)[..., None]

                    # sample noise to add to actions
                    noise = torch.randn(action_n.shape, device=device)
                    timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps, (B,), device=device).long()
                    noisy_actions = noise_scheduler.add_noise(action_n, noise, timesteps)

                    # predict the noise residual
                    noise_pred = noise_pred_net(noisy_actions, timesteps, global_cond=global_cond)

                    # L2 loss
                    loss = nn.functional.mse_loss(noise_pred, noise)

                    # optimize
                    loss.backward()
                    optimizer.step()
                    optimizer.zero_grad()
                    lr_scheduler.step()

                    ema.step(noise_pred_net)

                    # logging
                    loss_cpu = loss.item()
                    tdataloader.set_postfix(loss=loss_cpu)
                    epoch_loss.append(loss_cpu)
                    writer.add_scalar('Loss', loss_cpu, global_step=starting_step + len(all_losses) + len(epoch_loss))

            tglobal.set_postfix(loss=np.mean(epoch_loss))
            all_losses.extend(epoch_loss)
            writer.add_scalar('Loss/Epoch', np.mean(epoch_loss), global_step=epoch_idx)
except KeyboardInterrupt:
    all_losses.extend(epoch_loss)
    pass
writer.close()

In [ ]:
# Weights of the EMA model is used for inference
ema_noise_pred_net = ConditionalUnet1D(
    input_dim=action_dim,
    global_cond_dim=1,
)
ema_noise_pred_net.to(device)
ema.copy_to(ema_noise_pred_net.parameters())
torch.save(ema_noise_pred_net.state_dict(), f'{writer.log_dir}/ema_noise_pred_net.pth')
# torch.jit.save(torch.jit.script(ema_noise_pred_net), f'{writer.log_dir}/ema_noise_pred_net.pt')

# Print the log directory where the weights are saved
print(writer.log_dir)

# Plot the loss
plt.figure(figsize=(10, 3))
plt.semilogy(all_losses)
plt.title('Loss')

## Inference

In [ ]:
if True:
    # load pretrained weights
    # # This is default, training on 3000 drawings
    # folder1 = f'runs/Apr04_21-01-28_eagle'
    # # training on 100 drawings
    # folder1 = f'runs/Apr05_16-14-37_eagle'
    # # training with 512-length trajectories
    # folder1 = 'runs/Apr05_16-48-06_eagle'
    # # Fine-tuned with toppra dataset
    # folder1 = f'runs/May28_20-06-01_eagle'
    # Classifier-free guidance
    folder1 = f'runs/May28_20-34-26_eagle'

    device = torch.device('cuda')
    ema_noise_pred_net = ConditionalUnet1D(
        input_dim=action_dim,
        global_cond_dim=1,
    )
    ema_noise_pred_net.to(device)
    ema_noise_pred_net.load_state_dict(torch.load(f'{folder1}/ema_noise_pred_net.pth'))

In [ ]:
# B = 15  # num samples
# B = 6*6*4  # num samples
B = 6*6  # num samples
all_obs = {}
all_actions = {}
all_histories = {}
# global_cond = torch.linspace(0, 4, B // 4, device=device).reshape(-1, 1).repeat(1, 4).reshape(-1, 1)

# for horizon in tqdm([40, 80, 160, 320, 640, 1280, 2560]):
for horizon in tqdm([2560]):
    # action_n_init = torch.randn((B, pred_horizon, action_dim), device=device)
    action_n_init = torch.randn((B, horizon, action_dim), device=device)
    history = []

    action_n = network.eval(ema_noise_pred_net, noise_scheduler, action_n_init,
                            global_cond=torch.ones((B, 1), device=device) * 1.3,
                            # global_cond=global_cond,
                            log_history=history)

    action_n = action_n.detach().cpu().numpy()
    action = dataset2.unnormalize_action(action_n[..., -3:])

    all_actions[horizon] = action
    all_histories[horizon] = history
    all_obs[horizon] = dataset2.unnormalize_obs(action_n[..., :-3])

In [ ]:
scale = 1
# Plot trajectories
r, c = (B - 1) // 6 + 1, 6
fig, axes = plt.subplots(r, c, figsize=(12 / scale, 2.5 * r / scale))
fig.subplots_adjust(hspace=0.1, wspace=0.1)
acts = all_actions[40*64]
obss = all_obs[40*64]
# acts = all_actions[20]
print(acts.shape)
for act, obs, ax in zip(acts, obss, axes.flatten()):
    # ax.plot(*(obs - obs[0]).T, 'k.-')
    # plot_traj(ax, dataset2.normalize_action(act), obs=obs, markersize=1, linewidth=0.5, travel_kwargs=dict(linewidth=0.5))
    plot_traj(ax, dataset2.normalize_action(act), x0=obs[0], markersize=1, linewidth=0.5, travel_kwargs=dict(linewidth=0.5), clean=True)
    ax.axis('equal')
    # ax.set_xlim([0, 1])
    def zoom_lims(lims, factor):
        return (lims - np.mean(lims)) / factor + np.mean(lims)
    # ax.set_xlim(zoom_lims(ax.get_xlim(), 1.5))
    # ax.set_ylim(zoom_lims(ax.get_ylim(), 1.5))
fig.suptitle(f'T = {act.shape[0]}', fontsize=64 / scale);

In [ ]:
violations = []

def vel_and_acc(action_unnorm):
    DT = 0.02
    vel = action_unnorm[:, :, :2] / DT
    acc = torch.diff(vel, axis=1) / DT
    return vel, acc

acts = all_actions[40*64]
vels, accs = vel_and_acc(torch.from_numpy(acts))
for b, vel, acc in zip(global_cond, vels, accs):
    # print('{:.3f}, {:.3f}, {:.3f}'.format(
    #         b.item(),
    #         np.sum(np.abs(vel.cpu().numpy()) > 0.2) / vel.numpy().size,
    #         np.sum(np.abs(acc.cpu().numpy()) > 0.4) / acc.numpy().size))
    violations.append([
        b.item(),
        np.sum(np.abs(vel.cpu().numpy()) > 0.2) / vel.numpy().size,
        np.sum(np.abs(acc.cpu().numpy()) > 0.4) / acc.numpy().size
    ])

violations = np.array(violations)
# I want to apply "mean" for rows with matching "b".
violations_ = []
for b in sorted(np.unique(violations[:, 0])):
    mask = violations[:, 0] == b
    violations_.append([b, violations[mask, 1].mean(), violations[mask, 2].mean()])
violations = np.array(violations_)

fig = plt.figure(figsize=(6, 3))
ax = plt.gca()
ax2 = ax.twinx()
ax.plot(violations[:, 0], violations[:, 1], 'r.-')
ax2.plot(violations[:, 0], violations[:, 2], 'b.-')
ax.set_ylabel('Velocity', color='red')
ax2.set_ylabel('Acceleration', color='blue')
ax.set_xlabel('Class Scaling Factor')
plt.title('Fraction of Dynamics Violations vs Classifier-Free Guidance Scaling Factor', pad=20)
plt.tight_layout()
# fig.savefig('results/figs/classifier_free_violations_vs_factor.eps', dpi=300)

In [ ]:
scale = 1
# Plot trajectories
fig, axes = plt.subplots(3, 3, figsize=(10, 10))
acts = all_actions[40*64]
obss = all_obs[40*64]
# acts = all_actions[20]
print(acts.shape)
selected = [0, 3, 4, 6, 8, 9, 11, 14, 15]
ranges = [-1, -1, -1, -1, -1, -1, -1, -1, -1]
for act, obs, ax, rangee in zip(acts[selected], obss[selected], axes.flatten(), ranges):
    # ax.plot(*(obs - obs[0]).T, 'k.-')
    # plot_traj(ax, dataset2.normalize_action(act), obs=obs, markersize=1, linewidth=0.5, travel_kwargs=dict(linewidth=0.5))
    plot_traj(ax, dataset2.normalize_action(act[:rangee]), x0=obs[0], markersize=.5, linewidth=0.25, travel_kwargs=dict(linewidth=0.5), line_ls='k.-')
    ax.axis('equal')
    # ax.set_xlim([0, 1])
    def zoom_lims(lims, factor):
        return (lims - np.mean(lims)) / factor + np.mean(lims)
    # ax.set_xlim(zoom_lims(ax.get_xlim(), 1.5))
    # ax.set_ylim(zoom_lims(ax.get_ylim(), 1.5))
    # ax.axis('off')
    ax.grid(False)
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle('Approach 4: Classifier-Free Guidance Outputs', fontsize=32)
fig.set_tight_layout(True)
# fig.savefig('results/figs/classifier-free-1_drawings_black.eps')
# np.savez('results/figs/classifier-free-1_drawings_black.npz', acts=acts, obss=obss, selected=selected, ranges=ranges)

In [ ]:
# Plot velocities and accelerations

def vel_and_acc(action_unnorm):
    DT = 0.02
    vel = action_unnorm[:, :, :2] / DT
    acc = torch.diff(vel, axis=1) / DT
    return vel, acc

acts = all_actions[40*64]
vels, accs = vel_and_acc(torch.from_numpy(acts))
t = np.arange(0, act.shape[0]) * 0.02
vel, acc = vels[0], accs[0]

print(t.shape, vel.shape, acc.shape)

import matplotlib as mpl
new_color_cycle = ['r', 'g', 'b', 'k']
mpl.rcParams['axes.prop_cycle'] = mpl.cycler(color=new_color_cycle)

fig, axes = plt.subplots(2, 1, figsize=(6, 4), sharex=True)
axes[0].plot(t, vel, linewidth=1)
axes[1].plot(t[:-1], acc, linewidth=1)

axes[0].hlines([-.2, .2], t[0], t[-1], 'k', 'dashed')
axes[1].hlines([-.4, .4], t[0], t[-1], 'k', 'dashed')

axes[0].set_xlim(xmin=0)
axes[0].set_ylim(-.2 * 2, .2 * 2)
axes[1].set_ylim(-.4 * 2, .4 * 2)

fig.suptitle('Control Limits Adherance for Approach 4: Classifier-Free Guidance')
axes[1].set_xlabel('Time (s)')
axes[0].set_ylabel('Velocity (m/s)')
axes[1].set_ylabel('Acceleration (m/s)')
axes[0].legend(['x', 'y', 'limit'], loc='lower right')
fig.set_tight_layout(True)

# fig.savefig('results/figs/vel_acc-classifier-free-1.eps')
# np.savez('results/figs/vel_acc-classifier-free-1.npz', t=t, vel=vel, acc=acc)